In [ ]:
from datetime import datetime

job_id = session.sql("SELECT UUID_STRING()").collect()[0][0]
job_name = "DWH_DIM_MEDICATIONS_INFO_LOAD"
layer_name = "DWH"
start_time = datetime.now()

rows_loaded = 0
rows_failed = 0
run_status = "SUCCESS"
error_message = None

try:

    session.sql(f"""
    CREATE OR REPLACE TEMP TABLE TEMP_MEDICATIONS AS
    SELECT DISTINCT
        m.MEDICATION_CLASSIFICATION_CODE AS MEDICATION_CLASSIFICATION_CODE_AV_ID,
        av.VALUE_SHORT_DESC AS MEDICATION_CLASSIFICATION_CODE_DESC,
        m.DELETED_DATE,
        SHA2(CONCAT_WS('|',
            COALESCE(TO_VARCHAR(m.MEDICATION_CLASSIFICATION_CODE), ''),
            COALESCE(av.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(m.DELETED_DATE), '')
        )) AS CHECKSUM
    FROM {STG}.{STG_MEDICATIONS} m
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av
        ON m.MEDICATION_CLASSIFICATION_CODE = av.AV_ID
    WHERE m.MEDICATION_CLASSIFICATION_CODE IS NOT NULL
    """).collect()

    session.sql(f"""
    UPDATE {DWH}.{DIM_MEDICATIONS_INFO} tgt
    SET UPDATED_DATE = CURRENT_TIMESTAMP()
    FROM TEMP_MEDICATIONS src
    WHERE tgt.MEDICATION_CLASSIFICATION_CODE_AV_ID =
          src.MEDICATION_CLASSIFICATION_CODE_AV_ID
      AND tgt.UPDATED_DATE IS NULL
      AND tgt.CHECKSUM <> src.CHECKSUM
    """).collect()

    insert_result = session.sql(f"""
    INSERT INTO {DWH}.{DIM_MEDICATIONS_INFO}
    (
        MEDICATION_CLASSIFICATION_CODE_AV_ID,
        MEDICATION_CLASSIFICATION_CODE_DESC,
        CREATED_DATE,
        UPDATED_DATE,
        DELETED_DATE,
        CHECKSUM
    )

    SELECT
        src.MEDICATION_CLASSIFICATION_CODE_AV_ID,
        src.MEDICATION_CLASSIFICATION_CODE_DESC,
        CURRENT_TIMESTAMP(),
        NULL,
        src.DELETED_DATE,
        src.CHECKSUM

    FROM TEMP_MEDICATIONS src

    WHERE NOT EXISTS
    (
        SELECT 1
        FROM {DWH}.{DIM_MEDICATIONS_INFO} tgt
        WHERE tgt.MEDICATION_CLASSIFICATION_CODE_AV_ID =
              src.MEDICATION_CLASSIFICATION_CODE_AV_ID
          AND tgt.CHECKSUM = src.CHECKSUM
          AND tgt.UPDATED_DATE IS NULL
    )
    """).collect()

    rows_loaded = insert_result[0][0]

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"DIM_MEDICATIONS_INFO load completed successfully. Rows Loaded: {rows_loaded}"
    )

    print(f"DWH LOAD SUCCESS | Rows Loaded: {rows_loaded}")

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1
    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"DIM_MEDICATIONS_INFO load failed. Error: {error_message}"
    )

    print(f"DWH LOAD FAILED : {error_message}")